This is sequential workflow with llms. Although langgraph is not suitable to use here in linear workflows, but its just for practice.

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

load_dotenv()

In [ ]:
# 1. Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [ ]:
class chatbot_state(TypedDict):
    user_question:str
    llm_answer: str

In [ ]:
def get_answer(state:chatbot_state) -> chatbot_state:
    question = state['user_question']
    prompt = f"You are a hindi standup comedian. give answer in funny way, in hindi but english text. question is: {question}"
    answer = llm.invoke(prompt).content
    state['llm_answer'] = answer
    return state

In [ ]:
graph = StateGraph(chatbot_state)

# add nodes
graph.add_node("get_answer",get_answer)

# add edges
graph.add_edge(START,"get_answer")
graph.add_edge("get_answer",END)

# compile
workflow = graph.compile()

In [ ]:
# execute
initial_state = {"user_question":"Who is PM of Pakistan?"}

final_state = workflow.invoke(initial_state)
print(final_state['llm_answer'].content)


In [ ]:
# visualize
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())